# Loan Default Risk Predictor

This notebook is the canonical, end-to-end view of the project. It documents the LendingClub cohort, the leakage-aware input contract, the production preprocessing manifest, calibrated XGBoost scoring, threshold decisions, and batch scoring.

The default run is **artifact review mode**: it is fast, validates the checked-in production bundle, and scores representative raw applications. Set the `LOAN_RUN_REBUILD=1` environment variable before executing the notebook to run the leakage-safe temporal retraining workflow.

## Scope and responsible use

The target is `Charged Off` or `Default`. The output is decision support, not an approval engine, adverse-action notice, or substitute for legal, compliance, fairness, or credit-risk review. The model uses historical LendingClub data and must be independently validated before operational lending use.

`accepted_2007_to_2018Q4.csv` is the supervised-learning source because it contains loan outcomes.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata as package_metadata
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
RUN_REBUILD = os.environ.get('LOAN_RUN_REBUILD', '0') == '1'
RUN_SHAP = False
SAMPLE_ROWS = 5_000

def find_project_root():
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / 'models').is_dir() and (candidate / 'requirements.txt').is_file():
            return candidate
    raise FileNotFoundError('Could not locate the project root from ' + str(current))

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ACCEPTED_CSV = PROJECT_ROOT / 'accepted_2007_to_2018Q4.csv'
MODEL_PATH = PROJECT_ROOT / 'models' / 'xgb_calibrated_sigmoid.pkl'
METADATA_PATH = PROJECT_ROOT / 'models' / 'xgb_final_metadata.json'
MANIFEST_PATH = PROJECT_ROOT / 'models' / 'preprocessing.json'
FIGURES_DIR = PROJECT_ROOT / 'notebooks' / 'figures'

np.random.seed(SEED)
print('Project root:', PROJECT_ROOT)
print('Mode:', 'rebuild candidate' if RUN_REBUILD else 'production artifact review')
print('Large-data rebuild is opt-in:', RUN_REBUILD)

In [ ]:
def file_summary(path):
    path = Path(path)
    return {
        'file': str(path.relative_to(PROJECT_ROOT)),
        'exists': path.is_file(),
        'size_mb': round(path.stat().st_size / 1024**2, 2) if path.is_file() else None,
    }

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

inventory = pd.DataFrame([
    file_summary(ACCEPTED_CSV),
    file_summary(MODEL_PATH),
    file_summary(METADATA_PATH),
    file_summary(MANIFEST_PATH),
])
display(inventory)

version_names = ['numpy', 'pandas', 'scikit-learn', 'xgboost', 'joblib']
versions = []
for name in version_names:
    try:
        versions.append({'package': name, 'version': package_metadata.version(name)})
    except package_metadata.PackageNotFoundError:
        versions.append({'package': name, 'version': 'not installed'})
print('Python:', sys.version.split()[0], '| Platform:', platform.platform())
display(pd.DataFrame(versions))

accepted_columns = pd.read_csv(ACCEPTED_CSV, nrows=0).columns.tolist()
print('Accepted columns:', len(accepted_columns))

## 1. Load and validate the production bundle

The serving bundle contains the calibrated classifier, its threshold, and the versioned preprocessing manifest. The notebook implements the bundle contract locally, so it can be run as a standalone machine-learning project without the web application.

In [ ]:
import joblib

class InputValidationError(ValueError):
    pass

class ModelLoadError(RuntimeError):
    pass

class ModelPredictionError(RuntimeError):
    pass

class NotebookPreprocessor:
    def __init__(self, config):
        self.raw_features = list(config['raw_features'])
        self.numeric_features = list(config['numeric_features'])
        self.categorical_features = list(config['categorical_features'])
        self.model_features = list(config['model_features'])
        self.numeric_medians = {name: float(value) for name, value in config.get('numeric_medians', {}).items()}
        self.frequency_maps = {
            name: {str(key): float(value) for key, value in values.items()}
            for name, values in config.get('frequency_maps', {}).items()
        }
        self.annual_income_cap = float(config.get('annual_income_cap', 349956.835))
        self.credit_history_median = float(config.get('credit_history_median', 10.0))
        self.emp_length_median = float(config.get('emp_length_median', 6.0))
        self.feature_metadata = list(config.get('feature_metadata', []))
        if not self.raw_features or not self.model_features:
            raise ModelLoadError('Manifest must define raw and model features')
        if len(set(self.raw_features)) != len(self.raw_features):
            raise ModelLoadError('Manifest contains duplicate raw features')
        if len(set(self.model_features)) != len(self.model_features):
            raise ModelLoadError('Manifest contains duplicate model features')

    @staticmethod
    def is_missing(value):
        if value is None or value is pd.NA:
            return True
        if isinstance(value, str):
            return not value.strip()
        try:
            result = pd.isna(value)
        except (TypeError, ValueError):
            return False
        return isinstance(result, (bool, np.bool_)) and bool(result)

    @classmethod
    def parse_emp_length(cls, value):
        if cls.is_missing(value):
            return np.nan
        text = str(value).strip()
        if text == '< 1 year':
            return 0.0
        if text == '10+ years':
            return 10.0
        if text.endswith(' years') or text.endswith(' year'):
            try:
                return float(text.split()[0])
            except (TypeError, ValueError):
                return np.nan
        try:
            return float(text)
        except (TypeError, ValueError):
            return np.nan

    @classmethod
    def categorical_value(cls, value):
        return 'missing' if cls.is_missing(value) else str(value).strip()

    def transform(self, values):
        try:
            frame = values.copy() if isinstance(values, pd.DataFrame) else pd.DataFrame(values, dtype=object)
        except Exception as exc:
            raise InputValidationError('Input could not be converted to a table: ' + str(exc)) from exc
        if len(frame) == 0:
            return pd.DataFrame(columns=self.model_features, index=frame.index, dtype=float)
        if frame.columns.duplicated().any():
            duplicates = frame.columns[frame.columns.duplicated()].tolist()
            raise InputValidationError('Duplicate input feature names: ' + str(duplicates))
        extra = [name for name in frame.columns if name not in self.raw_features]
        if extra:
            raise InputValidationError('Invalid feature schema (unknown features: ' + str(extra) + ')')
        raw = frame.reindex(columns=self.raw_features).copy()
        transformed_columns = {}
        for name in self.numeric_features:
            original = raw[name]
            if original.map(lambda value: isinstance(value, (bool, np.bool_))).any():
                raise InputValidationError(
                    "Feature '%s' must contain numeric values, not booleans" % name
                )
            numeric = pd.to_numeric(original, errors='coerce')
            present = ~original.map(self.is_missing)
            invalid = present & numeric.isna()
            if invalid.any():
                raise InputValidationError(
                    "Feature '%s' has non-numeric values at rows %s" % (name, original.index[invalid].tolist())
                )
            try:
                finite = np.isfinite(numeric.dropna().to_numpy(dtype=float))
            except (TypeError, ValueError, OverflowError) as exc:
                raise InputValidationError("Feature '%s' must contain finite numbers" % name) from exc
            if not finite.all():
                raise InputValidationError("Feature '%s' has non-finite values" % name)
            if name == 'annual_inc':
                numeric = numeric.clip(upper=self.annual_income_cap)
            transformed_columns[name] = numeric.fillna(self.numeric_medians.get(name, 0.0))

        emp_length = raw['emp_length'].map(self.parse_emp_length)
        invalid_emp = (~raw['emp_length'].map(self.is_missing)) & emp_length.isna()
        if invalid_emp.any():
            raise InputValidationError(
                "Feature 'emp_length' has invalid values at rows %s" % raw.index[invalid_emp].tolist()
            )
        transformed_columns['emp_length_years'] = emp_length.fillna(self.emp_length_median)

        issue_values = raw['issue_d'].map(lambda value: np.nan if self.is_missing(value) else str(value).strip())
        earliest_values = raw['earliest_cr_line'].map(lambda value: np.nan if self.is_missing(value) else str(value).strip())
        issue_dates = pd.to_datetime(issue_values, format='%b-%Y', errors='coerce')
        earliest_dates = pd.to_datetime(earliest_values, format='%b-%Y', errors='coerce')
        invalid_issue = (~issue_values.map(self.is_missing)) & issue_dates.isna()
        invalid_earliest = (~earliest_values.map(self.is_missing)) & earliest_dates.isna()
        if invalid_issue.any() or invalid_earliest.any():
            fields = []
            if invalid_issue.any():
                fields.append('issue_d')
            if invalid_earliest.any():
                fields.append('earliest_cr_line')
            raise InputValidationError('Invalid date values for ' + ', '.join(fields) + '; expected Mon-YYYY')
        transformed_columns['credit_history_years'] = (
            (issue_dates - earliest_dates).dt.days / 365.25
        ).fillna(self.credit_history_median)

        for name, frequencies in self.frequency_maps.items():
            values_for_map = raw[name].map(self.categorical_value)
            transformed_columns[name + '_freq'] = values_for_map.map(frequencies).fillna(0.0)
        for name in self.categorical_features:
            values_for_dummy = raw[name].map(self.categorical_value)
            prefix = name + '_'
            for output_name in self.model_features:
                if output_name.startswith(prefix):
                    category = output_name[len(prefix):]
                    matches = values_for_dummy == category
                    if category != category.strip():
                        matches = matches | (values_for_dummy == category.strip())
                    transformed_columns[output_name] = matches.astype(float)

        transformed = pd.DataFrame(transformed_columns, index=raw.index)
        missing = [name for name in self.model_features if name not in transformed]
        if missing:
            raise InputValidationError('Preprocessing did not produce model features: ' + str(missing))
        transformed = transformed.loc[:, self.model_features].astype(float)
        if not np.isfinite(transformed.to_numpy()).all():
            raise InputValidationError('Transformed features contain non-finite values')
        return transformed

class NotebookModel:
    def __init__(self, model_path, metadata_path, manifest_path):
        self.model_file = Path(model_path).resolve()
        self.metadata_file = Path(metadata_path).resolve()
        self.manifest_file = Path(manifest_path).resolve()
        if not self.model_file.is_file() or not self.metadata_file.is_file() or not self.manifest_file.is_file():
            raise ModelLoadError('The model, metadata, and manifest files must exist')
        self.metadata = json.loads(self.metadata_file.read_text(encoding='utf-8'))
        self.manifest = json.loads(self.manifest_file.read_text(encoding='utf-8'))
        self.threshold = float(self.metadata['threshold'])
        if not np.isfinite(self.threshold) or not 0 <= self.threshold <= 1:
            raise ModelLoadError('Model threshold must be between 0 and 1')
        self.model = joblib.load(self.model_file)
        if not callable(getattr(self.model, 'predict_proba', None)):
            raise ModelLoadError('Loaded model does not provide predict_proba')
        self.preprocessor = NotebookPreprocessor(self.manifest)
        self.features = self.preprocessor.model_features
        self.input_features = self.preprocessor.raw_features
        self.feature_metadata = self.preprocessor.feature_metadata
        feature_count = getattr(self.model, 'n_features_in_', None)
        if feature_count is not None and feature_count != len(self.features):
            raise ModelLoadError('Model feature count does not match the manifest')
        classes = list(getattr(self.model, 'classes_', [0, 1]))
        if set(classes) != {0, 1} or len(classes) != 2:
            raise ModelLoadError('Model must have binary classes labeled 0 and 1')
        self.positive_class_index = classes.index(1)

    def predict_proba(self, values):
        try:
            frame = values.copy() if isinstance(values, pd.DataFrame) else pd.DataFrame(values, dtype=object)
        except Exception as exc:
            raise InputValidationError('Input could not be converted to a table: ' + str(exc)) from exc
        if len(frame) == 0:
            return pd.Series(index=frame.index, dtype=float, name='prob_default')
        extra = [name for name in frame.columns if name not in self.input_features]
        if extra:
            raise InputValidationError('Invalid feature schema (unknown features: ' + str(extra) + ')')
        prepared = self.preprocessor.transform(frame.reindex(columns=self.input_features))
        try:
            probabilities = np.asarray(self.model.predict_proba(prepared))
        except Exception as exc:
            raise ModelPredictionError('Model prediction failed: ' + str(exc)) from exc
        if probabilities.shape != (len(prepared), 2):
            raise ModelPredictionError('Model returned an invalid probability shape')
        try:
            probabilities = probabilities.astype(float)
        except (TypeError, ValueError, OverflowError) as exc:
            raise ModelPredictionError('Model returned non-numeric probabilities') from exc
        if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
            raise ModelPredictionError('Model returned probabilities outside 0 to 1')
        if not np.allclose(probabilities.sum(axis=1), 1):
            raise ModelPredictionError('Model probabilities do not sum to 1')
        return pd.Series(probabilities[:, self.positive_class_index], index=prepared.index, name='prob_default')

manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
production_wrapper = NotebookModel(MODEL_PATH, METADATA_PATH, MANIFEST_PATH)
bundle_metadata = production_wrapper.metadata
assert production_wrapper.features == manifest['model_features']
assert production_wrapper.input_features == manifest['raw_features']

bundle_summary = pd.Series({
    'model_version': bundle_metadata.get('model_version'),
    'training_cohort': bundle_metadata.get('training_cohort'),
    'probability_semantics': bundle_metadata.get('probability_semantics'),
    'threshold': production_wrapper.threshold,
    'raw_feature_count': len(production_wrapper.input_features),
    'model_feature_count': len(production_wrapper.features),
    'model_sha256': sha256_file(MODEL_PATH),
})
display(bundle_summary.to_frame('value'))

In [ ]:
raw_specs = pd.DataFrame(production_wrapper.feature_metadata)
raw_features = list(production_wrapper.input_features)
model_features = list(production_wrapper.features)

print('Raw application contract')
display(raw_specs[['name', 'type', 'placeholder', 'required']].head(20))
print('... total raw fields:', len(raw_features))
print('First model features:', model_features[:12])
print('Last model features:', model_features[-12:])
assert len(raw_features) == 86
assert len(model_features) == len(manifest['model_features'])
assert 'loan_status' not in model_features

## 2. Raw application preprocessing and scoring

The model accepts raw application-time fields. Omitted fields use the training manifest's imputation or missing-category behavior. Unknown fields are rejected. `int_rate` is expressed in percentage points, so 12% is sent as `12.0`.

In [ ]:
def example_application(wrapper):
    row = {spec['name']: spec.get('placeholder') for spec in wrapper.feature_metadata}
    row.update({
        'loan_amnt': 10_000,
        'funded_amnt': 10_000,
        'funded_amnt_inv': 10_000,
        'int_rate': 12.0,
        'annual_inc': 75_000,
        'dti': 15.0,
        'emp_length': '10+ years',
        'issue_d': 'Jan-2018',
        'earliest_cr_line': 'Jan-2000',
        'purpose': 'debt_consolidation',
        'grade': 'B',
        'sub_grade': 'B3',
        'home_ownership': 'RENT',
        'term': ' 36 months',
        'verification_status': 'Verified',
        'initial_list_status': 'w',
        'application_type': 'Individual',
        'disbursement_method': 'Cash',
        'addr_state': 'CA',
        'zip_code': '900xx',
    })
    return row

example_row = example_application(production_wrapper)

def score_rows(rows, wrapper=production_wrapper):
    probabilities = wrapper.predict_proba(rows)
    predictions = (probabilities >= wrapper.threshold).astype(int)
    return pd.DataFrame({
        'index': list(probabilities.index),
        'prob_default': probabilities.to_numpy(dtype=float),
        'pred_default': predictions.to_numpy(dtype=int),
    })

example_scores = score_rows([example_row, {}])
display(example_scores)
print('Decision threshold:', production_wrapper.threshold)

In [ ]:
transformed_example = production_wrapper.preprocessor.transform(pd.DataFrame([example_row]))
print('Transformed shape:', transformed_example.shape)
print('All finite:', np.isfinite(transformed_example.to_numpy()).all())
display(transformed_example.iloc[:, :12])
assert transformed_example.columns.tolist() == model_features
assert np.isfinite(transformed_example.to_numpy()).all()

In [ ]:
MAX_BATCH_SIZE = 1000

def score_batch(payload):
    if not isinstance(payload, dict) or not isinstance(payload.get('rows'), list):
        raise ValueError('rows must be an array of row objects')
    rows = payload['rows']
    if any(not isinstance(row, dict) for row in rows):
        raise ValueError('rows must be an array of row objects')
    if len(rows) > MAX_BATCH_SIZE:
        raise ValueError('rows cannot contain more than 1000 items')
    if not rows:
        return {'n': 0, 'results': []}
    scored = score_rows(rows)
    return {'n': len(scored), 'results': scored.to_dict(orient='records')}

batch_result = score_batch({'rows': [example_row]})
print(json.dumps(batch_result, indent=2))
print('Empty batch:', score_batch({'rows': []}))
try:
    score_batch({'rows': [{'unknown_field': 1}]})
except (InputValidationError, ValueError) as exc:
    print('Invalid input is rejected:', exc)

## 3. Data inventory, cohort definition, and leakage audit

The accepted CSV is about 1.6 GB, so review mode reads only its header and a small sample. The optional rebuild uses chunked reads, keeps terminal statuses (`Fully Paid`, `Charged Off`, `Default`), and defines the positive class as `Charged Off` or `Default`.

In [ ]:
sample = pd.read_csv(ACCEPTED_CSV, nrows=SAMPLE_ROWS, low_memory=False)
sample_issue = pd.to_datetime(sample['issue_d'], format='%b-%Y', errors='coerce')
sample_status = sample['loan_status'].value_counts(dropna=False).rename_axis('loan_status').to_frame('rows')
print('Accepted sample rows:', len(sample))
print('Sample issue range:', sample_issue.min(), 'to', sample_issue.max())
display(sample_status.head(12))

In [ ]:
application_feature_set = set(raw_features)
post_origination_tokens = (
    'hardship', 'settlement', 'recover', 'payment', 'pymnt', 'out_prncp',
    'last_pymnt', 'next_pymnt', 'last_credit_pull', 'last_fico'
)

def audit_reason(column):
    lower = column.lower()
    if column == 'loan_status':
        return 'drop: prediction target'
    if column in application_feature_set:
        if column == 'issue_d':
            return 'keep: cohort filter and credit-history feature'
        return 'keep: present in the frozen raw application contract'
    if any(token in lower for token in post_origination_tokens):
        return 'drop: post-origination or servicing information'
    return 'drop: not present in the exported application contract'

audit_df = pd.DataFrame({
    'column': accepted_columns,
    'decision': ['keep' if column in application_feature_set else 'drop' for column in accepted_columns],
})
audit_df['reason'] = audit_df['column'].map(audit_reason)
display(audit_df.groupby('decision').size().rename('columns').to_frame())
display(audit_df[audit_df['decision'] == 'drop'].head(25))
assert audit_df.loc[audit_df['column'].eq('loan_status'), 'decision'].eq('drop').all()
assert 'loan_status' not in model_features

In [ ]:
def load_training_cohort(path=ACCEPTED_CSV, chunksize=250_000):
    terminal_statuses = {'Fully Paid', 'Charged Off', 'Default'}
    usecols = list(dict.fromkeys(raw_features + ['loan_status']))
    frames = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize, low_memory=False):
        issue_dates = pd.to_datetime(chunk['issue_d'], format='%b-%Y', errors='coerce')
        eligible = issue_dates.ge(pd.Timestamp('2015-01-01'))
        eligible &= issue_dates.lt(pd.Timestamp('2019-01-01'))
        eligible &= chunk['loan_status'].isin(terminal_statuses)
        selected = chunk.loc[eligible].copy()
        if len(selected):
            selected['_issue_date'] = issue_dates.loc[eligible].to_numpy()
            selected['target'] = selected['loan_status'].isin({'Charged Off', 'Default'}).astype('int8')
            frames.append(selected)
    if not frames:
        raise ValueError('No eligible training rows were found')
    return pd.concat(frames, ignore_index=True)

if RUN_REBUILD:
    cohort = load_training_cohort()
    print('Rebuild cohort rows:', len(cohort))
    print('Positive rate:', round(cohort['target'].mean(), 4))
    print(cohort.groupby(cohort['_issue_date'].dt.year)['target'].agg(['size', 'mean']).to_string())
else:
    cohort = None
    print('Full cohort scan skipped. Set LOAN_RUN_REBUILD=1 to enable it.')

## 4. Leakage-safe temporal training

This section is deliberately opt-in. It uses four chronological periods: 2015 for model fitting, 2016 for probability calibration, 2017 for threshold selection, and 2018 for the final test. All learned preprocessing statistics are fitted on the model-fitting period only. The final bundle is written to a temporary run directory, validated, and then promoted to `models/`.

In [ ]:
if RUN_REBUILD:
    import shutil
    import time
    from sklearn.calibration import CalibratedClassifierCV
    from sklearn.frozen import FrozenEstimator
    from sklearn.metrics import (
        average_precision_score, brier_score_loss, f1_score, log_loss,
        precision_recall_curve, precision_score, recall_score, roc_auc_score,
    )
    from xgboost import XGBClassifier

    def fit_training_manifest(train_frame):
        config = {
            'version': 2,
            'training_cohort': 'accepted loans with terminal status, temporal release',
            'raw_features': list(raw_features),
            'numeric_features': list(manifest['numeric_features']),
            'categorical_features': list(manifest['categorical_features']),
            'feature_metadata': list(manifest['feature_metadata']),
        }
        train_only = train_frame.loc[:, raw_features].copy()
        numeric_medians = {}
        annual_income_cap = None
        for name in config['numeric_features']:
            numeric = pd.to_numeric(train_only[name], errors='coerce')
            if name == 'annual_inc':
                annual_income_cap = float(numeric.quantile(0.995))
                numeric = numeric.clip(upper=annual_income_cap)
            numeric_medians[name] = float(numeric.median())
        emp_length = train_only['emp_length'].map(NotebookPreprocessor.parse_emp_length)
        issue_values = train_only['issue_d'].map(
            lambda value: np.nan if NotebookPreprocessor.is_missing(value) else str(value).strip()
        )
        earliest_values = train_only['earliest_cr_line'].map(
            lambda value: np.nan if NotebookPreprocessor.is_missing(value) else str(value).strip()
        )
        issue_dates = pd.to_datetime(issue_values, format='%b-%Y', errors='coerce')
        earliest_dates = pd.to_datetime(earliest_values, format='%b-%Y', errors='coerce')
        credit_history = (issue_dates - earliest_dates).dt.days / 365.25
        config['numeric_medians'] = numeric_medians
        config['annual_income_cap'] = annual_income_cap
        config['emp_length_median'] = float(emp_length.median())
        config['credit_history_median'] = float(credit_history.median())
        config['frequency_maps'] = {}
        for name in ('addr_state', 'zip_code'):
            values = train_only[name].map(NotebookPreprocessor.categorical_value)
            config['frequency_maps'][name] = values.value_counts(normalize=True).to_dict()

        observed = {}
        for name in config['categorical_features']:
            observed[name] = set(train_only[name].map(NotebookPreprocessor.categorical_value))
        derived = set(config['numeric_features']) | {'emp_length_years', 'credit_history_years'}
        derived |= {name + '_freq' for name in config['frequency_maps']}
        model_feature_order = []
        for feature in manifest['model_features']:
            if feature in derived:
                model_feature_order.append(feature)
                continue
            for name in config['categorical_features']:
                prefix = name + '_'
                if feature.startswith(prefix):
                    category = feature[len(prefix):]
                    if category in observed[name] or category.strip() in observed[name]:
                        model_feature_order.append(feature)
                    break
        config['model_features'] = model_feature_order
        return config

    def metrics_at_threshold(y_true, probabilities, threshold):
        predictions = (probabilities >= threshold).astype(int)
        return {
            'roc_auc': float(roc_auc_score(y_true, probabilities)),
            'pr_auc': float(average_precision_score(y_true, probabilities)),
            'log_loss': float(log_loss(y_true, probabilities, labels=[0, 1])),
            'brier_score': float(brier_score_loss(y_true, probabilities)),
            'precision': float(precision_score(y_true, predictions, zero_division=0)),
            'recall': float(recall_score(y_true, predictions, zero_division=0)),
            'f1': float(f1_score(y_true, predictions, zero_division=0)),
            'flagged_rate': float(predictions.mean()),
            'threshold': float(threshold),
        }

    def select_validation_threshold(y_true, probabilities):
        precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
        f1_values = (2 * precision * recall) / (precision + recall + 1e-12)
        if len(thresholds) == 0:
            return 0.5
        best_index = int(np.nanargmax(f1_values[:-1]))
        return float(thresholds[best_index])

    split_starts = {
        'train': pd.Timestamp('2015-01-01'),
        'calibration': pd.Timestamp('2016-01-01'),
        'validation': pd.Timestamp('2017-01-01'),
        'test': pd.Timestamp('2018-01-01'),
        'end': pd.Timestamp('2019-01-01'),
    }
    split_masks = {
        'train': cohort['_issue_date'].ge(split_starts['train']) & cohort['_issue_date'].lt(split_starts['calibration']),
        'calibration': cohort['_issue_date'].ge(split_starts['calibration']) & cohort['_issue_date'].lt(split_starts['validation']),
        'validation': cohort['_issue_date'].ge(split_starts['validation']) & cohort['_issue_date'].lt(split_starts['test']),
        'test': cohort['_issue_date'].ge(split_starts['test']) & cohort['_issue_date'].lt(split_starts['end']),
    }
    if not all(mask.any() for mask in split_masks.values()):
        raise ValueError('Every temporal split must contain at least one row')
    split_summary = pd.DataFrame({
        name: {
            'rows': int(mask.sum()),
            'positive': int(cohort.loc[mask, 'target'].sum()),
            'positive_rate': float(cohort.loc[mask, 'target'].mean()),
            'start': str(cohort.loc[mask, '_issue_date'].min().date()),
            'end': str(cohort.loc[mask, '_issue_date'].max().date()),
        }
        for name, mask in split_masks.items()
    }).T
    display(split_summary)

    train_raw = cohort.loc[split_masks['train']]
    calibration_raw = cohort.loc[split_masks['calibration']]
    validation_raw = cohort.loc[split_masks['validation']]
    test_raw = cohort.loc[split_masks['test']]
    train_manifest = fit_training_manifest(train_raw)
    train_preprocessor = NotebookPreprocessor(train_manifest)
    X_train = train_preprocessor.transform(train_raw[raw_features])
    X_calibration = train_preprocessor.transform(calibration_raw[raw_features])
    X_validation = train_preprocessor.transform(validation_raw[raw_features])
    X_test = train_preprocessor.transform(test_raw[raw_features])
    y_train = train_raw['target'].astype(int)
    y_calibration = calibration_raw['target'].astype(int)
    y_validation = validation_raw['target'].astype(int)
    y_test = test_raw['target'].astype(int)
    assert X_train.columns.tolist() == train_manifest['model_features']
    assert X_calibration.columns.tolist() == X_train.columns.tolist()
    assert X_validation.columns.tolist() == X_train.columns.tolist()
    assert X_test.columns.tolist() == X_train.columns.tolist()

    negative = int((y_train == 0).sum())
    positive = int((y_train == 1).sum())
    model_parameters = {
        'n_estimators': 200,
        'max_depth': 6,
        'learning_rate': 0.05,
        'subsample': 0.9,
        'colsample_bytree': 0.9,
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'scale_pos_weight': negative / max(positive, 1),
        'random_state': SEED,
        'n_jobs': -1,
    }
    base_model = XGBClassifier(**model_parameters)
    started = time.perf_counter()
    base_model.fit(X_train, y_train)
    training_seconds = time.perf_counter() - started
    calibrated_model = CalibratedClassifierCV(FrozenEstimator(base_model), method='sigmoid')
    calibrated_model.fit(X_calibration, y_calibration)
    validation_probability = calibrated_model.predict_proba(X_validation)[:, 1]
    test_probability = calibrated_model.predict_proba(X_test)[:, 1]
    selected_threshold = select_validation_threshold(y_validation, validation_probability)
    validation_metrics = metrics_at_threshold(y_validation, validation_probability, selected_threshold)
    test_metrics = metrics_at_threshold(y_test, test_probability, selected_threshold)
    training_metrics = {'validation': validation_metrics, 'test': test_metrics}
    print('Training seconds:', round(training_seconds, 2))
    print('Validation metrics:', validation_metrics)
    print('Test metrics:', test_metrics)

    release_dir = PROJECT_ROOT / 'notebooks' / 'runs' / 'temporal_release'
    release_dir.mkdir(parents=True, exist_ok=True)
    release_model_path = release_dir / 'xgb_calibrated_sigmoid.pkl'
    release_manifest_path = release_dir / 'preprocessing.json'
    release_metadata_path = release_dir / 'xgb_final_metadata.json'
    joblib.dump(calibrated_model, release_model_path)
    release_manifest_path.write_text(json.dumps(train_manifest, indent=2), encoding='utf-8')
    release_metadata = {
        'model_path': 'models/xgb_calibrated_sigmoid.pkl',
        'preprocessing': 'preprocessing.json',
        'model_version': 'xgb-calibrated-sigmoid-temporal-v2',
        'threshold': selected_threshold,
        'threshold_objective': 'maximum F1 on the 2017 validation period',
        'training_cohort': 'accepted loans issued 2015-01-01 through 2018-12-01 with terminal status',
        'label_definition': 'Charged Off or Default versus Fully Paid',
        'label_maturity_note': 'Only terminal statuses in the source are used; no fixed outcome horizon is available.',
        'probability_semantics': 'estimated probability of Charged Off or Default',
        'split_definition': {name: {key: value for key, value in row.items()} for name, row in split_summary.to_dict(orient='index').items()},
        'calibration': {'method': 'sigmoid', 'fit_period': '2016 calendar year', 'estimator': 'FrozenEstimator'},
        'preprocessing_fit_period': '2015 calendar year only',
        'model_parameters': model_parameters,
        'metrics': training_metrics,
        'raw_feature_count': len(train_manifest['raw_features']),
        'model_feature_count': len(train_manifest['model_features']),
        'data_sha256': sha256_file(ACCEPTED_CSV),
    }
    release_metadata['model_sha256'] = sha256_file(release_model_path)
    release_metadata['manifest_sha256'] = sha256_file(release_manifest_path)
    release_metadata_path.write_text(json.dumps(release_metadata, indent=2), encoding='utf-8')
    candidate_wrapper = NotebookModel(release_model_path, release_metadata_path, release_manifest_path)
    candidate_probability = candidate_wrapper.predict_proba([example_row]).iloc[0]
    assert 0 <= candidate_probability <= 1
    for source, destination in ((release_model_path, MODEL_PATH), (release_manifest_path, MANIFEST_PATH), (release_metadata_path, METADATA_PATH)):
        shutil.copy2(source, destination)
    production_wrapper = NotebookModel(MODEL_PATH, METADATA_PATH, MANIFEST_PATH)
    manifest = production_wrapper.manifest
    bundle_metadata = production_wrapper.metadata
    raw_features = list(production_wrapper.input_features)
    model_features = list(production_wrapper.features)
    raw_specs = pd.DataFrame(production_wrapper.feature_metadata)
    example_row = example_application(production_wrapper)
    example_scores = score_rows([example_row, {}])
    transformed_example = production_wrapper.preprocessor.transform(pd.DataFrame([example_row]))
    print('Promoted model:', METADATA_PATH)
else:
    training_metrics = None
    print('Temporal training skipped. Set LOAN_RUN_REBUILD=1 to run it.')

In [ ]:
if RUN_REBUILD:
    display(pd.DataFrame(training_metrics).T)
else:
    print('No new metrics were generated. Set LOAN_RUN_REBUILD=1 to run temporal training.')

In [ ]:
if RUN_SHAP:
    import matplotlib.pyplot as plt
    import shap

    shap_raw = pd.read_csv(ACCEPTED_CSV, usecols=raw_features, nrows=min(SAMPLE_ROWS, 1_000), low_memory=False)
    shap_frame = production_wrapper.preprocessor.transform(shap_raw)
    calibrated = production_wrapper.model
    estimators = getattr(calibrated, 'calibrated_classifiers_', [])
    if not estimators:
        raise RuntimeError('The calibrated artifact does not expose a base estimator for SHAP')
    estimator = getattr(estimators[0], 'estimator', None)
    if estimator is None:
        estimator = getattr(estimators[0], 'base_estimator', None)
    if estimator is None:
        raise RuntimeError('Could not locate the calibrated base estimator')
    shap_values = shap.TreeExplainer(estimator)(shap_frame)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    shap.summary_plot(shap_values, shap_frame, show=False, max_display=30)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'notebook_shap_summary.png', dpi=150)
    plt.close()
    print('Saved notebook_shap_summary.png')
else:
    print('Fresh SHAP computation skipped. Existing SHAP artifacts are listed above.')

In [ ]:
# Lightweight acceptance checks for the notebook's user-visible flow.
assert 0 <= production_wrapper.threshold <= 1
assert len(production_wrapper.input_features) == 86
assert len(production_wrapper.features) == len(manifest['model_features'])
assert example_scores['prob_default'].between(0, 1).all()
assert set(example_scores['pred_default']).issubset({0, 1})
assert score_batch({'rows': []}) == {'n': 0, 'results': []}
if 'model_sha256' in bundle_metadata:
    assert bundle_metadata['model_sha256'] == sha256_file(MODEL_PATH)
if 'manifest_sha256' in bundle_metadata:
    assert bundle_metadata['manifest_sha256'] == sha256_file(MANIFEST_PATH)

try:
    production_wrapper.predict_proba([{'not_a_feature': 1}])
except InputValidationError as exc:
    print('Unknown raw fields rejected:', str(exc))
else:
    raise AssertionError('Unknown raw fields were accepted')

print('Notebook acceptance checks passed.')

## 5. Final summary and limitations

The production path is: raw application fields → deterministic manifest transform → model features → calibrated XGBoost probability → validation-selected threshold decision. The retraining path keeps preprocessing, calibration, threshold selection, and final testing in separate chronological periods.

The remaining limitation is the source label: this release uses terminal statuses from the source file and does not have a fixed outcome horizon. A production release should add a defined observation window and subgroup stability/fairness evaluation before operational lending use.

In [ ]:
final_report = pd.Series({
    'model_version': bundle_metadata.get('model_version'),
    'target': 'Charged Off or Default',
    'cohort': bundle_metadata.get('training_cohort'),
    'raw_features': len(raw_features),
    'model_features': len(model_features),
    'threshold': production_wrapper.threshold,
    'accepted_data_file': str(ACCEPTED_CSV.relative_to(PROJECT_ROOT)),
    'rebuild_run': RUN_REBUILD,
    'production_model_sha256': sha256_file(MODEL_PATH),
})
display(final_report.to_frame('value'))
print('Complete: the notebook validates production scoring and contains a leakage-safe temporal training path.')